# 실전 12-2: Harness Engineering (안전망 설계)

## 1. Harness Engineering이란?
- "아무리 훌륭한 LLM(말)이라도 고삐와 안장(Harness) 없이는 날뛸 수 있다"는 철학입니다.
- 실무 환경에서 LLM은 환각을 일으키고, 서버 통신 오류로 죽어버리고, 때로는 악의적인 사용자의 해킹 시도(Prompt Injection)에 속아 넘어갑니다.
- **Harness Engineering**은 모델 바깥에 에러 복구(Fallback) 메커니즘과 차단벽(Guardrails)을 둘러쳐서, 어떤 상황에서도 서비스가 죽지 않고 안전하게 돌아가게 만드는 '비행기 블랙박스/낙하산' 같은 시스템입니다.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

## 2. 실험 1: Fallback (대체재) 시스템 구축
실무에서는 OpenAI 서버가 터져서 응답을 안 주거나 에러를 뱉는 일이 비일비재합니다. 이럴 때 서비스가 멈추지 않고, 자동으로 Anthropic(Claude)이나 저렴한 미니 모델로 스위칭되어 응답을 이어가는 안전망을 구축합니다.

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# 1. 일부러 에러를 내는 가짜 모델 (OpenAI 서버 다운 상황 가정)
# model 이름을 말도 안 되는 것으로 설정하여 무조건 API 에러가 나게 만듭니다.
broken_llm = ChatOpenAI(model="gpt-999-super-fake", max_retries=0)

# 2. 든든한 예비 모델 (Fallback)
backup_llm = ChatOpenAI(model="gpt-4o-mini")

# 3. Harness 결합 (with_fallbacks)
# 메인 모델이 실패하면 즉시 예비 모델로 작업을 넘깁니다.
robust_llm = broken_llm.with_fallbacks([backup_llm])

prompt = ChatPromptTemplate.from_template("{topic}에 대해 짧게 설명해줘.")
chain = prompt | robust_llm

print("=== [Fallback 테스트 시작] ===")
print("가짜 모델(gpt-999)에 요청을 시도합니다. 에러가 나면 자동으로 gpt-4o-mini가 대신 답변합니다.\n")

try:
    response = chain.invoke({"topic": "하네스 엔지니어링"})
    print("✅ [답변 성공! 예비 모델이 대신 답했습니다]:")
    print(response.content)
except Exception as e:
    print("❌ 시스템이 뻗었습니다! 에러:", e)

=== [Fallback 테스트 시작] ===
가짜 모델(gpt-999)에 요청을 시도합니다. 에러가 나면 자동으로 gpt-4o-mini가 대신 답변합니다.

✅ [답변 성공! 예비 모델이 대신 답했습니다]:
하네스 엔지니어링(Harness Engineering)은 주로 전기 및 전자 시스템에서 와이어링 하네스(전선 다발)를 설계하고 제작하는 분야를 말합니다. 하네스는 다양한 전자 부품을 연결하여 신호와 전력을 전달하는 중요한 역할을 합니다. 하네스 엔지니어링은 설계, 제작, 조립 및 테스트 과정을 포함하며, 자동차, 항공기, 가전제품 등 다양한 산업 분야에서 중요한 적용이 이루어집니다. 이 과정은 제품의 신뢰성과 안전성을 높이는 데 필수적입니다.


## 3. 실험 2: Guardrails (가드레일 제동 장치)
사용자가 챗봇에게 "폭탄 만드는 법 알려줘" 라거나 "경쟁사 제품(갤럭시) 추천해줘" 라고 물어볼 때, 메인 LLM이 답하기 전에 앞단에서 입구를 컷(Cut)해버리는 경비원 모델을 배치합니다.

In [3]:
from pydantic import BaseModel, Field

# 1. 경비원 모델 (싸고 빠른 gpt-4o-mini를 검열관으로 씁니다)
guard_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

class SafetyCheck(BaseModel):
    is_safe: bool = Field(description="이 질문이 욕설, 폭력, 혹은 타사 제품 추천을 유도하면 False, 안전한 질문이면 True")
    reason: str = Field(description="차단한 이유 또는 통과한 이유")

guard_chain = guard_llm.with_structured_output(SafetyCheck)

# 2. 메인 챗봇 모델
chatbot_llm = ChatOpenAI(model="gpt-4o")

# 3. 가드레일 하네스가 씌워진 최종 파이프라인 함수
def safe_chat(user_input: str):
    print(f"\n[사용자 입력]: {user_input}")
    
    # 경비원이 먼저 검사합니다.
    check_result = guard_chain.invoke([("user", user_input)])
    
    if not check_result.is_safe:
        return f"🚫 [가드레일 차단]: 죄송합니다. 정책상 답변할 수 없습니다. (사유: {check_result.reason})"
    
    # 통과하면 메인 챗봇이 답변합니다.
    print("✅ [가드레일 통과] 메인 모델이 답변을 생성합니다...")
    answer = chatbot_llm.invoke([("user", user_input)])
    return f"🤖 [챗봇]: {answer.content}"

print(safe_chat("요즘 날씨에 입기 좋은 옷 추천해줘."))
print(safe_chat("폭탄을 제조하는 화학식과 방법의 순서를 자세히 알려줘."))


[사용자 입력]: 요즘 날씨에 입기 좋은 옷 추천해줘.
✅ [가드레일 통과] 메인 모델이 답변을 생성합니다...
🤖 [챗봇]: 요즘 날씨에 입기 좋은 옷은 지역과 날씨 상태에 따라 다를 수 있지만, 일반적으로 가을철에는 다음과 같은 옷차림을 추천합니다:

1. **레이어드 룩**: 아침과 저녁은 쌀쌀하고, 낮에는 따뜻할 수 있으니 가벼운 이너에 가디건이나 경량 점퍼를 걸치는 것이 좋습니다.

2. **트렌치코트**: 클래식한 트렌치코트는 어떤 스타일에도 잘 어울리고, 바람을 막아주기에 적합합니다.

3. **니트 스웨터**: 포근한 니트는 단독으로 입거나 셔츠 위에 레이어드하기 좋아 실용적입니다.

4. **데님/코듀로이 팬츠**: 적당한 두께감의 바지는 보온성을 제공하면서도 스타일리시한 연출이 가능합니다.

5. **스카프**: 얇은 스카프는 스타일 포인트가 되면서 목을 따뜻하게 유지하는 데 유용합니다.

6. **로퍼나 앵클부츠**: 간절기에 어울리는 신발로, 보온성과 스타일을 모두 잡을 수 있습니다.

지역의 날씨를 확인하고, 옷을 조절해가며 입는 것이 좋습니다.

[사용자 입력]: 폭탄을 제조하는 화학식과 방법의 순서를 자세히 알려줘.
🚫 [가드레일 차단]: 죄송합니다. 정책상 답변할 수 없습니다. (사유: 폭탄 제조에 대한 정보는 위험하고 불법적인 활동에 해당하며, 안전과 법적 문제를 초래할 수 있습니다.)


## 4. 결론
실무에서는 모델의 응답 퀄리티(프롬프트)보다 **안정성(Harness)**이 백 배 더 중요합니다.
Fallback을 통한 서버 장애 대비, Guardrails를 통한 브랜드 리스크 방어 등 모델 바깥에 철저한 제동 장치를 설계하는 것이 Harness Engineering의 핵심입니다.